# X5_P2 — Investigación exploratoria vs. precio

Segundo notebook de **X5_alt** (versión paralela y simplificada de X5 — ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md), sección 9, y la
especificación completa en
[`docs/plans/solicitud_claude_code_X5_P2_notebook_v2.md`](../docs/plans/solicitud_claude_code_X5_P2_notebook_v2.md)).

Reemplaza a `X5_P2.py` (solo cargaba la tabla e imprimía dimensiones/rango) como
implementación oficial de P2: un notebook exhaustivo de investigación descriptiva y,
en segundo plano, prospectiva, sobre la tabla maestra generada por `X5_P1.ipynb`. Las
secciones de análisis (auditoría, variable vs. precio, ceteris paribus, regímenes,
etc.) se agregan de forma incremental en TODOs siguientes, respetando el orden
narrativo del plan. Este primer paso deja la tabla maestra cargada — todo el análisis
posterior parte de ella.

**Input único que se puede cambiar:**

In [1]:
valor = 'BTCUSD'  # inicialmente solo BTCUSD

## 0. Preparación

Importamos `pandas` y reutilizamos `config.py` para la ruta de `resources/x5_alt/`
(misma convención que `X5_P1.ipynb`).

In [2]:
import sys
from pathlib import Path

import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activo: {valor}")

Activo: BTCUSD


## 1. Cargar tabla maestra (X5_P1)

Input obligatorio de P2: `resources/x5_alt/{valor}_tabla_maestra.csv`, generado por
`X5_P1.ipynb`. P2 no recalcula nada de P1 — si el archivo no existe, hay que correr
`X5_P1.ipynb` primero para ese activo.

In [3]:
path_tabla = cfg.CARPETA_X5_ALT / f'{valor}_tabla_maestra.csv'
if not path_tabla.exists():
    raise FileNotFoundError(
        f"No existe {path_tabla}. Corré X5_P1.ipynb para {valor} antes de X5_P2."
    )

tabla = pd.read_csv(path_tabla, parse_dates=['DateTime'])

print(f"Tabla maestra cargada: {valor} — {tabla.shape[0]} filas x {tabla.shape[1]} columnas")
print(f"Rango: {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")
tabla.head()

Tabla maestra cargada: BTCUSD — 40283 filas x 37 columnas
Rango: 2021-10-27 16:00:00 → 2026-06-02 21:00:00


,DateTime,Open,High,Low,Close,Tick_Volume,Spread,Real_Volume,x3_sma_20,x3_sma_dist_20,...,x3_roc_20,x3_vol_24h,x3_vol_7d,x3_drawdown_20,x3_drawdown_50,x3_trend_slope_20,x3_trend_slope_50,x2_score,x2_score_cross,x2_score_tendencia
0,2021-10-27 16:00:00,58734.06,59042.92,58532.74,58856.56,1347.0,586,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-10-27 17:00:00,58856.56,59166.83,58831.57,58942.47,2065.0,1402,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-10-28 09:00:00,60431.05,61276.66,60360.70,61048.85,2631.0,3977,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-10-28 10:00:00,61048.79,61231.67,60814.46,61042.45,4085.0,3349,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-10-28 11:00:00,61041.32,61106.04,60732.14,60999.73,3942.0,681,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Variables disponibles para el análisis (mapeo mínimo)

Antes de entrar al núcleo de P2 (variable vs. precio) necesitamos saber exactamente
qué columnas son `x3_*`, `x2_*` u "otra" variable explicativa relevante, y detectar
de entrada cuáles no tienen datos utilizables. Esto **no** es la auditoría completa
de la tabla maestra (esa es un TODO aparte, con cobertura, primera/última
observación, tipos y frecuencia efectiva en detalle) — es solo lo mínimo para poder
iterar correctamente en la sección siguiente sin llegar tarde al análisis central.

Se excluyen del análisis `Open`/`High`/`Low`/`Close` (son el precio mismo, no una
variable explicativa), `Spread` (artefacto de microestructura del bróker, no una
señal macro) y `Real_Volume` (viene siempre en 0 en los CSV de MT5 — ver
`CLAUDE.md`, sección del algoritmo de soportes). Se incluye `Tick_Volume` como
variable "Otra": la relación volumen-precio es una pregunta clásica que vale la
pena mirar aunque no venga de X2/X3.

In [ ]:
EXCLUIR = {'DateTime', 'Open', 'High', 'Low', 'Close', 'Spread', 'Real_Volume'}

FAMILIAS = {}
for col in tabla.columns:
    if col in EXCLUIR:
        continue
    if col.startswith('x3_'):
        FAMILIAS[col] = 'Técnico'
    elif col.startswith('x2_'):
        FAMILIAS[col] = 'Fundamental'
    else:
        FAMILIAS[col] = 'Otra'

cobertura = tabla[list(FAMILIAS)].notna().mean().mul(100).round(2).sort_values()
variables_utilizables = [c for c in FAMILIAS if tabla[c].notna().sum() > 30]
variables_sin_datos = [c for c in FAMILIAS if c not in variables_utilizables]

n_tecnico = sum(v == 'Técnico' for v in FAMILIAS.values())
n_fundamental = sum(v == 'Fundamental' for v in FAMILIAS.values())
n_otra = sum(v == 'Otra' for v in FAMILIAS.values())
print(f"Variables candidatas: {len(FAMILIAS)} ({n_tecnico} técnicas, {n_fundamental} fundamentales, {n_otra} otra)")
print(f"Utilizables para el análisis vs. precio: {len(variables_utilizables)}")
print(f"Sin datos suficientes (se excluyen del loop): {variables_sin_datos}")
print()
cobertura.to_frame('% filas no nulas')

## 3. Núcleo descriptivo — variable vs. precio (Nivel 2)

**Pregunta rectora:** cuando cada variable (`x3_*`, `x2_*` y `Tick_Volume`) tomó
determinados valores en el pasado, ¿cómo se encontraba el precio de BTC en ese
mismo instante? Es el bloque más importante de P2 — ver
`docs/plans/solicitud_claude_code_X5_P2_notebook_v2.md`, secciones 4.6 y 14.1 — y
por eso aparece muy arriba en el notebook, justo después del mapeo mínimo de
variables.

**Qué se calcula para cada variable:**
- Serie temporal normalizada (z-score) junto al precio, para comparar la forma de
  dos series con escalas distintas en un mismo eje.
- Scatter variable vs. precio, con una regresión lineal simple (descriptiva, no
  predictiva).
- Pearson (relación lineal) y Spearman (relación monótona, no necesariamente
  lineal): comparar ambos dice si la relación es aproximadamente lineal (valores
  similares) o si hay curvatura que Pearson subestima (Spearman notoriamente
  mayor).
- Kendall (tau) como medida adicional de asociación: compara pares de
  observaciones en vez de magnitudes, por lo que es más robusto a outliers que
  Pearson/Spearman — relevante acá porque el precio de BTC tiene eventos extremos
  (rallies, crashes) que pueden inflar una correlación de magnitudes.
- Precio medio por decil de la variable (cuantiles/bins): permite ver si la
  relación es aproximadamente monótona a lo largo del rango de valores, sin asumir
  que es lineal.

**Cómo interpretar dirección y fuerza:** cada variable se etiqueta como
positiva/negativa y nula-despreciable (\|r\| < 0.10) / débil (< 0.30) / moderada
(< 0.50) / fuerte (< 0.70) / muy fuerte (≥ 0.70), usando Pearson como referencia.
Es una convención práctica para poder ordenar variables rápido, no un estándar
estadístico universal.

**Ejemplo para fijar la intuición:** si Pearson(RSI, Precio) ≈ 0.04, esto **no**
significa que el RSI sea una variable inútil — es un oscilador acotado entre 0 y
100 que, por diseño, no debería moverse junto al *nivel* del precio (BTC a 20.000 y
BTC a 100.000 pueden tener RSI = 50 por igual). La pregunta relevante para un
oscilador como el RSI suele estar en otro lado (momentum, retornos), no en el nivel
de precio — se anota como matiz de lectura, no como falla del análisis.

**Limitaciones (aplican a todo este bloque):**
- Correlación no implica causalidad.
- Una relación contemporánea (`X_t` vs. `Precio_t`) no implica capacidad
  predictiva sobre el precio futuro — eso es la Parte II (prospectiva), fuera de
  esta sección.
- Con ~40.000 observaciones horarias fuertemente autocorrelacionadas (no
  independientes), los p-values salen extremadamente pequeños incluso para
  relaciones triviales. No deben leerse como si vinieran de una muestra
  independiente — se muestran solo como referencia bajo ese supuesto simplificado.
- Este bloque no cubre todavía no linealidades más allá de lo que revela el
  gráfico de deciles, ni estabilidad temporal de la relación (por año/rolling) —
  son TODOs separados y posteriores en `docs/tracking/todos.md` (sección
  `X5_alt`).
- `x2_score`, `x2_score_cross` y `x2_score_tendencia` están 100% vacíos en la tabla
  actual (hallazgo ya documentado en `docs/plans/X5_alternativo.md`, sección 8) y
  se excluyen del loop siguiente — no hay nada que correlacionar todavía.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

PRECIO_COL = 'Close'
N_BINS = 10


def zscore(serie: pd.Series) -> pd.Series:
    """Normaliza a media 0 / desvío 1 para comparar series de escalas distintas en un mismo eje."""
    return (serie - serie.mean()) / serie.std()


def interpretar_fuerza(r: float) -> str:
    """Etiqueta dirección/fuerza de Pearson según la convención práctica descrita arriba."""
    signo = 'positiva' if r >= 0 else 'negativa'
    a = abs(r)
    if a < 0.10:
        fuerza = 'nula/despreciable'
    elif a < 0.30:
        fuerza = 'débil'
    elif a < 0.50:
        fuerza = 'moderada'
    elif a < 0.70:
        fuerza = 'fuerte'
    else:
        fuerza = 'muy fuerte'
    return f"{signo} {fuerza}"


def analizar_variable_vs_precio(df: pd.DataFrame, col: str, familia: str,
                                 precio_col: str = PRECIO_COL, n_bins: int = N_BINS) -> dict | None:
    """Genera el bloque completo (3 gráficos + métricas) de variable vs. precio para una columna."""
    sub = df[[col, precio_col]].dropna()
    if len(sub) < 30:
        print(f"{col}: solo {len(sub)} filas no nulas — se omite del análisis.")
        return None

    x, y = sub[col].to_numpy(), sub[precio_col].to_numpy()
    pear_r, pear_p = scipy_stats.pearsonr(x, y)
    spear_r, spear_p = scipy_stats.spearmanr(x, y)
    kend_r, kend_p = scipy_stats.kendalltau(x, y)

    try:
        bins = pd.qcut(sub[col], n_bins, duplicates='drop')
    except ValueError:
        bins = pd.qcut(sub[col].rank(method='first'), n_bins)
    precio_por_decil = sub.groupby(bins, observed=True)[precio_col].mean()
    precio_por_decil.index = [f"b{i+1}" for i in range(len(precio_por_decil))]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(
        f"{col}  [{familia}]  —  Pearson={pear_r:.3f}  Spearman={spear_r:.3f}  Kendall={kend_r:.3f}"
        f"  ({interpretar_fuerza(pear_r)})",
        fontsize=11,
    )

    ax = axes[0]
    ax.plot(df['DateTime'], zscore(df[precio_col]), label=precio_col, color='#1f77b4', linewidth=0.8)
    ax.plot(df['DateTime'], zscore(df[col]), label=col, color='#d62728', linewidth=0.8, alpha=0.8)
    ax.set_title('Serie temporal normalizada (z-score)')
    ax.set_ylabel('z-score')
    ax.legend(fontsize=8)

    ax = axes[1]
    ax.scatter(x, y, s=2, alpha=0.15, color='#1f77b4')
    coef = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='#d62728', linewidth=1.5)
    ax.set_title('Scatter vs. precio + regresión lineal')
    ax.set_xlabel(col)
    ax.set_ylabel(precio_col)

    ax = axes[2]
    ax.bar(precio_por_decil.index, precio_por_decil.values, color='#2ca02c', alpha=0.85)
    ax.set_title(f'{precio_col} medio por decil de {col}')
    ax.set_ylabel(f'{precio_col} medio')
    ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return {
        'variable': col, 'familia': familia, 'n': len(sub),
        'pearson': pear_r, 'pearson_p': pear_p,
        'spearman': spear_r, 'spearman_p': spear_p,
        'kendall': kend_r, 'kendall_p': kend_p,
        'direccion_fuerza': interpretar_fuerza(pear_r),
    }

In [ ]:
resultados_vs_precio = []
for col in variables_utilizables:
    resultado = analizar_variable_vs_precio(tabla, col, FAMILIAS[col])
    if resultado is not None:
        resultados_vs_precio.append(resultado)

for col in variables_sin_datos:
    print(f"{col} [{FAMILIAS[col]}]: excluido — 0% de cobertura, ver limitación anotada arriba.")

### Ranking inicial de relaciones

Tabla de apoyo para escanear rápido los resultados anteriores, ordenada por
\|Pearson\| descendente. **No** es todavía la síntesis descriptiva completa (esa
combina además frecuencia efectiva, régimen, estabilidad y redundancia — TODO
separado y posterior) ni debe leerse como un ranking predictivo: es solo el
ranking de qué tan fuerte se ha relacionado, históricamente y en contemporáneo,
cada variable con el *nivel* del precio.

In [ ]:
ranking_inicial = (
    pd.DataFrame(resultados_vs_precio)
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values('abs_pearson', ascending=False)
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_inicial

## 4. Ceteris paribus histórico — regresión multivariable controlando por otras variables (Nivel 3)

**Qué se analiza:** un modelo lineal múltiple que explica el precio contemporáneo
(`Close_t`) con TODAS las variables utilizables de la Sección 2 al mismo tiempo
(`variables_utilizables` — `x3_*` + `Tick_Volume`; los `x2_*` siguen sin datos, ver
limitación de la Sección 3):

`Close_t = β₀ + β₁·X1_t + β₂·X2_t + ... + ε_t`

**Por qué:** la Sección 3 midió cada variable contra el precio *una a la vez*. Muchos
técnicos comparten la misma información (varias medias móviles, `roc_10`/`roc_20`,
`drawdown_20`/`drawdown_50`, etc.), así que una correlación cruda alta puede ser la
misma señal "contada dos veces" a través de dos columnas distintas. La regresión
múltiple estima el efecto de cada variable *manteniendo las demás fijas* — la
definición operacional de "ceteris paribus" en un modelo lineal.

**Metodología reutilizada:** mismo enfoque manual con `numpy`/`scipy` (mínimos
cuadrados + errores estándar clásicos de OLS, sin `statsmodels`) ya usado en
`Otros/scripts/X5_analisis_exploratorio.ipynb` (`regresion_ceteris_paribus`/
`graficar_regresion`), copiado y adaptado acá al target `Close` y al universo
`x3_*`/`Tick_Volume` de este notebook — no se importa desde `Otros/` para no acoplar
X5_alt a la implementación de X5 original. Todas las variables (incluido el target)
se estandarizan a z-score, así el coeficiente de cada una es el cambio esperado en
el precio (en desvíos estándar) cuando esa variable sube 1 desvío estándar, con las
demás fijas.

**Multicolinealidad:** con ~27 indicadores técnicos, es esperable que varios estén
altamente correlacionados entre sí (ej. `x3_sma_20` vs. `x3_sma_50` vs.
`x3_sma_200`). Se mide con el **Factor de Inflación de Varianza (VIF)** de cada
variable, calculado como la diagonal de la inversa de la matriz de correlación de
las variables estandarizadas (equivalente a `1/(1-R²)` de regresar esa variable
contra todas las demás, sin correrlas una por una). VIF alto (convención: > 10)
señala que el coeficiente de esa variable es inestable/poco confiable
individualmente — no implica eliminarla, sino leer su coeficiente aislado con
cautela.

**Sensibilidad:** se compara la especificación completa contra una reducida que
excluye las variables con VIF > 10, para ver si el signo o la magnitud de los
coeficientes remanentes cambia de forma importante — un cambio grande indicaría que
el resultado depende de qué variables colineales se incluyen, no solo del target.

**Cómo interpretar:** igual que la Sección 3 (dirección/fuerza por magnitud del
coeficiente estandarizado), más una columna nueva: si el signo del coeficiente
ceteris paribus coincide con el signo del Pearson crudo de la Sección 3. Cuando no
coincide (o el coeficiente pierde significancia), es exactamente el tipo de
relación que "desaparece al controlar por otras variables" que pide la
especificación de P2.

**Limitaciones:**
- Las de la Sección 3 (autocorrelación temporal → p-values no confiables bajo el
  supuesto de independencia; no implica causalidad; no implica capacidad
  predictiva).
- Un modelo lineal aditivo no captura interacciones ni no linealidades — eso es la
  sección de no linealidades (TODO separado y posterior).
- Con variables muy colineales, el vector de coeficientes puede estar mal
  identificado aunque el R² total sea razonable — por eso el chequeo de VIF y la
  especificación reducida son parte obligatoria de esta sección, no un adicional
  opcional.
- Igual que la Sección 3, este bloque se deja **sin ejecutar en Mac** — la tabla
  maestra de esta copia está incompleta/desactualizada (ver `docs/plans/
  X5_alternativo.md`, sección 8). Correr y agregar la celda de interpretación de
  hallazgos corresponde hacerlo en Windows, con la tabla maestra real.

In [ ]:
def calcular_vif(df: pd.DataFrame, cols: list) -> pd.Series:
    """VIF de cada columna vía la diagonal de la inversa de su matriz de correlación
    (equivalente a 1/(1-R²) de regresarla contra las demás, sin correr N regresiones)."""
    sub = df[cols].dropna()
    corr = sub.corr(numeric_only=True).to_numpy()
    vif = np.diag(np.linalg.pinv(corr))
    return pd.Series(vif, index=cols, name='VIF').sort_values(ascending=False)


def regresion_ceteris_paribus(df: pd.DataFrame, feature_cols: list, target_col: str,
                               min_filas: int = 30) -> pd.DataFrame | None:
    """Regresión lineal múltiple con todas las variables estandarizadas (z-score).
    El coeficiente de cada variable es el cambio esperado en el target (en desvíos
    estándar) cuando esa variable sube 1 desvío estándar, manteniendo fijas las demás
    — la definición operacional de "efecto ceteris paribus" en un modelo lineal.
    Calculado a mano (mínimos cuadrados + errores estándar de OLS) para no depender
    de statsmodels — mismo enfoque de `Otros/scripts/X5_analisis_exploratorio.ipynb`."""
    cols = [c for c in feature_cols if c in df.columns]
    sub = df[cols + [target_col]].dropna()
    n = len(sub)
    if n < min_filas:
        print(f"  (!) Solo {n} filas sin NaN (mínimo sugerido {min_filas}). Se omite la regresión.")
        return None

    X = sub[cols].to_numpy(dtype=float)
    y = sub[target_col].to_numpy(dtype=float)

    sd_x = X.std(axis=0, ddof=0)
    cols_validas = sd_x > 1e-12
    if not cols_validas.all():
        descartadas = [c for c, ok in zip(cols, cols_validas) if not ok]
        print(f"  (!) Columnas sin varianza (constantes) descartadas de la regresión: {descartadas}")
    cols = [c for c, ok in zip(cols, cols_validas) if ok]
    X = X[:, cols_validas]

    mu_x, sd_x = X.mean(axis=0), X.std(axis=0, ddof=0)
    Xz = (X - mu_x) / sd_x
    sd_y = y.std(ddof=0)
    sd_y = sd_y if sd_y > 1e-12 else 1.0
    yz = (y - y.mean()) / sd_y

    X_dis = np.column_stack([np.ones(n), Xz])
    beta, *_ = np.linalg.lstsq(X_dis, yz, rcond=None)
    y_hat = X_dis @ beta
    resid = yz - y_hat

    p = X_dis.shape[1]
    dof = n - p
    if dof <= 0:
        print("  (!) No quedan grados de libertad. Se omite la regresión.")
        return None
    sigma2 = (resid @ resid) / dof
    XtX_inv = np.linalg.pinv(X_dis.T @ X_dis)
    se = np.sqrt(np.clip(np.diag(sigma2 * XtX_inv), 0, None))
    t_val = np.divide(beta, se, out=np.zeros_like(beta), where=se > 0)
    p_val = 2 * scipy_stats.t.sf(np.abs(t_val), df=dof)

    ss_tot = ((yz - yz.mean()) ** 2).sum()
    r2 = 1 - (resid @ resid) / ss_tot if ss_tot > 1e-12 else np.nan

    tabla = pd.DataFrame({
        'feature': ['(intercepto)'] + cols,
        'coef_estandarizado': beta,
        'error_estandar': se,
        't': t_val,
        'p_valor': p_val,
    })
    tabla = tabla[tabla['feature'] != '(intercepto)'].reset_index(drop=True)
    tabla['significativo_5pct'] = tabla['p_valor'] < 0.05
    tabla = tabla.reindex(tabla['coef_estandarizado'].abs().sort_values(ascending=False).index).reset_index(drop=True)
    tabla.attrs['r2'] = r2
    tabla.attrs['n'] = n
    tabla.attrs['dof'] = dof
    return tabla


def graficar_regresion(tabla: pd.DataFrame, titulo: str) -> None:
    if tabla is None:
        return
    fig, ax = plt.subplots(figsize=(7.5, max(2.2, 0.4 * len(tabla))))
    orden = tabla.iloc[::-1]
    colores = ['#2a9d8f' if v >= 0 else '#e76f51' for v in orden['coef_estandarizado']]
    ax.barh(orden['feature'], orden['coef_estandarizado'],
            xerr=1.96 * orden['error_estandar'], color=colores, capsize=3)
    ax.axvline(0, color='#333333', linewidth=0.8)
    ax.set_xlabel('Coeficiente estandarizado (± IC 95%) — efecto ceteris paribus')
    ax.set_title(f"{titulo}\nR²={tabla.attrs.get('r2', float('nan')):.3f}  n={tabla.attrs.get('n', '?')}")
    plt.tight_layout()
    plt.show()

In [ ]:
tabla_reg_precio = regresion_ceteris_paribus(tabla, variables_utilizables, PRECIO_COL, min_filas=100)
if tabla_reg_precio is not None:
    graficar_regresion(tabla_reg_precio, f"{valor} — efecto ceteris paribus sobre {PRECIO_COL} (especificación completa)")
tabla_reg_precio

### Multicolinealidad — VIF de las variables

VIF de cada variable en `variables_utilizables`, calculado sobre la matriz de
correlación completa. Regla práctica: VIF > 10 se considera colinealidad alta (su
coeficiente individual en la regresión de arriba es poco confiable), sin que eso
implique eliminar la variable — solo leer su coeficiente con cautela.

In [ ]:
vif_precio = calcular_vif(tabla, variables_utilizables)
vars_vif_alto = vif_precio[vif_precio > 10].index.tolist()
print(f"Variables con VIF > 10 (colinealidad alta): {len(vars_vif_alto)} de {len(variables_utilizables)}")
vif_precio.to_frame('VIF')

### Sensibilidad — especificación reducida (sin colinealidad alta)

Se repite la regresión excluyendo las variables con VIF > 10 y se comparan los
coeficientes de las que sobreviven en ambas especificaciones. Un coeficiente que
cambia de signo o pierde gran parte de su magnitud al pasar de la especificación
completa a la reducida es una señal de que su efecto "ceteris paribus" depende
fuertemente de qué colineales se incluyen — no un resultado robusto.

In [ ]:
variables_reducidas = [c for c in variables_utilizables if c not in vars_vif_alto]
tabla_reg_reducida = regresion_ceteris_paribus(tabla, variables_reducidas, PRECIO_COL, min_filas=100)
if tabla_reg_reducida is not None:
    graficar_regresion(tabla_reg_reducida,
                        f"{valor} — efecto ceteris paribus sobre {PRECIO_COL} (especificación reducida, sin VIF > 10)")

if tabla_reg_precio is not None and tabla_reg_reducida is not None:
    comparacion_specs = (
        tabla_reg_precio[['feature', 'coef_estandarizado']]
        .merge(tabla_reg_reducida[['feature', 'coef_estandarizado']], on='feature',
               suffixes=('_completa', '_reducida'))
    )
    comparacion_specs['cambia_signo'] = (
        np.sign(comparacion_specs['coef_estandarizado_completa'])
        != np.sign(comparacion_specs['coef_estandarizado_reducida'])
    )
else:
    comparacion_specs = None
    print("  (!) Falta alguna de las dos regresiones — no se puede comparar especificaciones.")
comparacion_specs

### Comparación con la correlación cruda (Sección 3)

Cruza el coeficiente ceteris paribus (especificación completa) con el Pearson
crudo de la Sección 3, para identificar qué variables mantienen signo/fuerza al
controlar por las demás y cuáles ven su relación con el precio debilitada o
invertida — la pregunta explícita de la especificación de P2 ("¿qué relaciones
desaparecen al controlar por otras variables?").

In [ ]:
if tabla_reg_precio is not None:
    comparacion_cruda_vs_cp = (
        ranking_inicial[['variable', 'familia', 'pearson']]
        .merge(tabla_reg_precio[['feature', 'coef_estandarizado', 'significativo_5pct']],
               left_on='variable', right_on='feature')
        .drop(columns='feature')
    )
    comparacion_cruda_vs_cp['mismo_signo'] = (
        np.sign(comparacion_cruda_vs_cp['pearson']) == np.sign(comparacion_cruda_vs_cp['coef_estandarizado'])
    )
    comparacion_cruda_vs_cp = (
        comparacion_cruda_vs_cp
        .sort_values('coef_estandarizado', key=abs, ascending=False)
        .reset_index(drop=True)
    )
else:
    comparacion_cruda_vs_cp = None
comparacion_cruda_vs_cp